# Inference code for SWIN-Base

In [1]:
cd "/home/simon/Documents/Zwischen Wörtern und Pixeln/"

/home/simon/Documents/Zwischen Wörtern und Pixeln


## Get modules

In [2]:
import argparse
import os
from time import perf_counter
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from safetensors.torch import load_file
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass

In [3]:
#import custom Config
from Config import cfg

#device
cfg.device = torch.device(cfg.device)
print(cfg.device)

cuda


## Definitions

`load_model`: Loads a model checkpoint, preferrably in a .safetensor file format. <br>
> **Note:** Loading a .pth checkpoint is a security risk and discouraged. The .safetensor file format is much safer.

<br>

Will already cast precision, either **BF16** for faster inference or **FP32** for maximum accuracy. BF16 is preferred over FP16 as it is closer to the FP32 mantissa/exponent strucutre.

In [4]:
def load_model(checkpoint_path: str, precision: str, device: torch.device) -> torch.nn.Module:
    """Load Swin-Base from a .safetensors checkpoint.

    Args:
        checkpoint_path : path to .safetensors file
        precision       : "fp32" or "bf16"
        device          : torch device

    Returns:
        model in eval mode at the requested precision
    """
    print(f"Loading model from: {checkpoint_path}")

    model = timm.create_model(
        cfg.model_name,
        pretrained  = False, #weights come from the checkpoint
        num_classes = cfg.num_classes,
        drop_rate   = cfg.drop_rate, #must match training config
        drop_path_rate = cfg.drop_path_rate, #must match training config
    )

    #load safetensors weights
    #checkpoint is loaded on cpu first and later moved to gpu for compatibility reasons
    state_dict = load_file(checkpoint_path, device = "cpu")
    missing, unexpected = model.load_state_dict(state_dict, strict = True)

    if missing:
        print(f"  Warning — missing keys  : {missing}")
    if unexpected:
        print(f"  Warning — unexpected keys: {unexpected}")

    #cast precision
    if precision == "bf16":
        if not torch.cuda.is_bf16_supported():
            print("  Warning: BF16 not supported on this GPU, falling back to FP32.")
            precision = "fp32"
        else:
            model = model.to(torch.bfloat16)
            print("  Precision: BF16")
    else:
        print("  Precision: FP32")

    model = model.to(device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Model: {cfg.model_name}  |  {total_params:.1f} M params  |  device: {device}")
    
    return model, precision

<br><br>
We resize all images to 224x224px and apply ImageNet normalisations on all images to be inferenced on. <br>
No augmentation will be conducted as augmentation is only useful in training.
<br><br>

In [5]:
def get_inference_transform() -> A.Compose:
    """Validation-equivalent transform — resize + ImageNet normalisation only.
    No augmentation. Must match the val transform used during training.
    """
    return A.Compose([
        
        A.Resize(cfg.img_size, 
                 cfg.img_size),
        
        A.Normalize(mean = cfg.imagenet_mean, 
                    std = cfg.imagenet_std),
        
        ToTensorV2(),
    ])

In [6]:
class ImageDataset(Dataset):
    """Minimal inference dataset — no labels, just images and their paths.

    Skips unreadable files gracefully and records them in self.failed.
    """

    def __init__(self, image_paths: list[str], transform: A.Compose):
        self.transform   = transform
        self.valid_paths = []
        self.failed      = []

        for p in image_paths:
            img = cv2.imread(p)
            if img is None:
                self.failed.append(p)
            else:
                self.valid_paths.append(p)

        if self.failed:
            print(f"  Warning: {len(self.failed)} image(s) could not be read and will be skipped:")
            for f in self.failed[:5]:
                print(f"    {f}")
            if len(self.failed) > 5:
                print(f"    ... and {len(self.failed) - 5} more")

    def __len__(self) -> int:
        return len(self.valid_paths)

    def __getitem__(self, idx: int):
        path  = self.valid_paths[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, path

<br><br>
Image paths are collected from a directory. <br>
Will raise exceptions if image file type is unsupported or if there are no images in the specified directory.
<br><br>

In [7]:
#define acceptable file types
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

def collect_image_paths(input_path: str) -> list[str]:
    """Accept a single image file or a directory (non-recursive)."""
    p = Path(input_path)
    
    if p.is_file():
        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            raise ValueError(f"Unsupported file extension: {p.suffix}")
        return [str(p)]
    
    elif p.is_dir():
        paths = sorted(
            str(f) for f in p.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not paths:
            raise FileNotFoundError(f"No images found in directory: {input_path}")
        return paths
    
    else:
        raise FileNotFoundError(f"Input path does not exist: {input_path}")

<br><br>
Inference is conducted over all image paths. <br>
The function will force either BF16 or FP32 precision. Based on hardware, only FP32 might be available. <br>
Device and batch size are both specified in `Config`. <br>
Model will cast one prediction. Confidence is provided for all classes. Should confidence be below a threshold to be specified in `Config`, the model will cast "Unknown" as label. <br>

> **Note:** High confidence does not necessarily translate to a correct prediction. The model can be very confident and still be wrong.

<br><br>

In [8]:
@torch.no_grad()
def run_inference(
    model       : torch.nn.Module,
    loader      : DataLoader,
    device      : cfg.device,
    precision   : str,
    threshold   : float,
) -> list[dict]:
    """Run batched inference. Returns a list of result dicts.
    Batch size is specified in config.

    Each dict contains:
        image_path   : absolute path to the image
        prediction   : predicted emotion label (or "Unknown" if below threshold)
        confidence   : probability of the predicted class
        Optimistic   : probability score
        Pessimistic  : probability score
        Hostile      : probability score
        Neutral      : probability score
    """
    results    = []

    #allow only BF16 and FP32
    autocast_dtype = torch.bfloat16 if precision == "bf16" else torch.float32

    for images, paths in tqdm(loader, desc = "Inferencing", unit = "batch"):
        images = images.to(device)

        #cast input to model precision
        if precision == "bf16":
            images = images.to(torch.bfloat16)

        with torch.amp.autocast("cuda", dtype = autocast_dtype, enabled = (device.type == "cuda")):
            logits = model(images)

        #always compute probabilities in FP32 for numerical accuracy
        probs = F.softmax(logits.float(), dim = 1).cpu()
        preds = probs.argmax(dim = 1)
        confs = probs.max(dim = 1).values

        for path, pred, conf, prob in zip(paths, preds, confs, probs):
            pred_idx  = pred.item()
            conf_val  = conf.item()

            #apply confidence threshold, label as Unknown if not confident enough
            if conf_val < cfg.threshold:
                label = "Unknown"
            else:
                label = cfg.EMOTION_NAMES[pred_idx]

            results.append({
                "image_path":  path,
                "prediction":  label,
                "confidence":  round(conf_val, 4),
                **{name: round(prob[i].item(), 4) for i, name in enumerate(cfg.EMOTION_NAMES)},
            })

    return results

<br><br>
Summary counts and elapsed time are printed for review.
<br><br>

In [9]:
def print_summary(results: list[dict], elapsed: float) -> None:
    """Print a summary table to stdout."""
    total    = len(results)
    unknown  = sum(1 for r in results if r["prediction"] == "Unknown")
    counts   = {name: sum(1 for r in results if r["prediction"] == name)
                for name in cfg.EMOTION_NAMES}

    print(f"\n{'-' * 58}")
    print(f"  Results  {total} images  |  {elapsed:.1f}s  |  "
          f"{total/elapsed:.0f} img/s  |  {precision}")
    print(f"{'-' * 58}")
    
    for name in cfg.EMOTION_NAMES:
        pct = counts[name] / total * 100
        bar = " " * int(pct / 2)
        print(f"  {name:<14} {counts[name]:>5}  ({pct:5.1f}%)  {bar}")
    if unknown:
        pct = unknown / total * 100
        print(f"  {'Unknown':<14} {unknown:>5}  ({pct:5.1f}%)  (below threshold ({cfg.threshold}))")
    print(f"{'-' * 58}\n")

    #show a few example predictions
    print("  Sample predictions:")
    for r in results[:5]:
        name = Path(r["image_path"]).name
        print(f"    {name:<40} → {r['prediction']:<14} ({r['confidence']:.3f})")
    if len(results) > 5:
        print(f"    ... and {len(results) - 5} more\n")

## Inference

We use the SWIN-Base model for inference. In domain adaptation we reached these performance metrics after 41 epochs: <br>


- Cohen's Kappa: 0.7057
- Macro-F1 (unweighted): 0.7748
- Validation loss: 0.5957


We inference at single-precision (FP32) on our NVIDIA Blackwell GB203 accelerator.

In [10]:
#load best checkpoint
model, precision = load_model("adaptation_SWIN-Base_checkpoint_final.safetensors",
           
                              "FP32", #higher FP32 precision
          
                              cfg.device)

Loading model from: adaptation_SWIN-Base_checkpoint_final.safetensors
  Precision: FP32
  Model: swin_base_patch4_window7_224  |  86.7 M params  |  device: cuda


In [11]:
transforms = get_inference_transform()

In [12]:
from pprint import pprint

#get image paths
images = collect_image_paths("Sample_Dataset/Reichel/images_with_faces")

#verify
pprint(images[:5])
print(len(images))

['Sample_Dataset/Reichel/images_with_faces/1001169_523480_image.jpeg',
 'Sample_Dataset/Reichel/images_with_faces/1001183_523496_image.jpeg',
 'Sample_Dataset/Reichel/images_with_faces/1001227_523515_image.jpeg',
 'Sample_Dataset/Reichel/images_with_faces/1001245_523536_image.jpeg',
 'Sample_Dataset/Reichel/images_with_faces/1001338_523635_image.jpeg']
4450


<br><br>
The dataset is built to be used in the loader. <br>
Loader will define batches in size as defined in `Config`. <br>
Number of workers defines how many subprocesses for data fetching are spawned. It is also specified in `Config` according to hardware limitations. <br>
Our system has 16 CPU-cores and 64 Gigabytes of DRAM. It therefore supports a high number of workers. Depending on hardware it can be better to choose a low number for workers.
<br><br>

In [13]:
dataset = ImageDataset(images, transforms)

In [15]:
#define dataloader
loader = DataLoader(
    dataset,
    batch_size = cfg.batch_size_main,
    shuffle = False,
    num_workers = cfg.num_workers,
    pin_memory = True
)

<br><br>
We finally conduct the actual inference run. <br><br>
`model`, `loader` and `precision` objects have been specified above. <br>
`device` and `threshold` are specified in `Config`.
<br><br>

In [16]:
#run inference
start_time = perf_counter()

results = run_inference(
    model = model,
    loader = loader,
    device = cfg.device,
    precision = precision,
    threshold = cfg.threshold
)

end_time = perf_counter()

elapsed = end_time - start_time

Inferencing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 70/70 [00:11<00:00,  6.17batch/s]


In [17]:
print_summary(results, elapsed)


----------------------------------------------------------
  Results  4450 images  |  11.3s  |  392 img/s  |  FP32
----------------------------------------------------------
  Optimistic      1488  ( 33.4%)                  
  Pessimistic      338  (  7.6%)     
  Hostile          664  ( 14.9%)         
  Neutral         1850  ( 41.6%)                      
  Unknown          110  (  2.5%)  (below threshold (0.4))
----------------------------------------------------------

  Sample predictions:
    1001169_523480_image.jpeg                → Hostile        (0.996)
    1001183_523496_image.jpeg                → Neutral        (0.540)
    1001227_523515_image.jpeg                → Neutral        (0.850)
    1001245_523536_image.jpeg                → Neutral        (0.953)
    1001338_523635_image.jpeg                → Optimistic     (0.978)
    ... and 4445 more



In [18]:
df_results = pd.DataFrame(results)

df_results.head()

,image_path,prediction,confidence,Optimistic,Pessimistic,Hostile,Neutral
0,Sample_Dataset/Reichel/images_with_faces/10011...,Hostile,0.9959,0.0015,0.0000,0.9959,0.0026
1,Sample_Dataset/Reichel/images_with_faces/10011...,Neutral,0.5398,0.4582,0.0012,0.0009,0.5398
2,Sample_Dataset/Reichel/images_with_faces/10012...,Neutral,0.8501,0.0354,0.0419,0.0726,0.8501
3,Sample_Dataset/Reichel/images_with_faces/10012...,Neutral,0.9528,0.0197,0.0083,0.0193,0.9528
4,Sample_Dataset/Reichel/images_with_faces/10013...,Optimistic,0.9782,0.9782,0.0108,0.0038,0.0072


## Save results to csv

In [19]:
df_results.to_csv("inference_results_SWIN-base_fp32.csv",
                 encoding = "UTF-8",
                 index = False)